In [1]:
import os
from huggingface_hub import login
from dotenv import load_dotenv
load_dotenv()

/home/ubuntu/MultiModal_Video_Model/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
HF_TOKEN =  os.getenv("HF_TOKEN") # Replace with your token from https://huggingface.co/settings/tokens
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
import torch
from diffusers import HunyuanVideoPipeline
from diffusers.utils import export_to_video

assert torch.cuda.is_available(), "CUDA is required — no GPU detected!"

num_gpus = torch.cuda.device_count()
total_vram = 0
sm_major = torch.cuda.get_device_properties(0).major

print(f"{'GPU':>5}  {'Name':<30}  {'VRAM':>8}  {'SM':>5}")
print("-" * 56)
for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    vram_gb = props.total_memory / 1e9
    total_vram += vram_gb
    print(f"{i:>5}  {props.name:<30}  {vram_gb:>7.1f}G  {props.major}.{props.minor:>3}")

per_gpu_mem = total_vram / num_gpus
dtype = torch.float16 if sm_major < 8 else torch.bfloat16

torch.backends.cudnn.benchmark = True

print(f"\n{'='*56}")
print(f"GPUs: {num_gpus}  |  Total VRAM: {total_vram:.0f} GB  |  dtype: {dtype}")
print(f"{'='*56}")


  GPU  Name                                VRAM     SM
--------------------------------------------------------
    0  NVIDIA H100 PCIe                   85.0G  9.  0

GPUs: 1  |  Total VRAM: 85 GB  |  dtype: torch.bfloat16


In [4]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if num_gpus > 1:
    from diffusers import HunyuanVideoTransformer3DModel
    from transformers import LlamaModel
    from accelerate import dispatch_model

    print(f"🚀 {num_gpus} GPUs ({total_vram:.0f} GB total) — splitting model layers across all GPUs")

    # ── GPU partition ──────────────────────────────────────────────
    # Transformer (~26 GB fp16, 60 blocks): GPUs 0-4  ~5 GB each
    # Text encoder (~16 GB fp16, LLaMA):    GPUs 5-6  ~8 GB each
    # VAE + CLIP (small):                   GPU 7
    # ──────────────────────────────────────────────────────────────

    # Load transformer — accelerate "auto" splits layers across GPUs 0-4
    transformer = HunyuanVideoTransformer3DModel.from_pretrained(
        "hunyuanvideo-community/HunyuanVideo",
        subfolder="transformer",
        torch_dtype=dtype,
        device_map="auto",
        max_memory={0: "7GiB", 1: "7GiB", 2: "7GiB", 3: "7GiB", 4: "7GiB"},
        token=HF_TOKEN,
    )

    # Load text encoder — split layers across GPUs 5-6
    text_encoder = LlamaModel.from_pretrained(
        "hunyuanvideo-community/HunyuanVideo",
        subfolder="text_encoder",
        torch_dtype=dtype,
        device_map="auto",
        max_memory={5: "10GiB", 6: "10GiB"},
        token=HF_TOKEN,
    )

    # Build pipeline with pre-loaded components (skips re-loading them)
    pipe = HunyuanVideoPipeline.from_pretrained(
        "hunyuanvideo-community/HunyuanVideo",
        transformer=transformer,
        text_encoder=text_encoder,
        torch_dtype=dtype,
        token=HF_TOKEN,
    )

    # Dispatch small components to GPU 7 (hooks auto-transfer inputs)
    dispatch_model(pipe.text_encoder_2, device_map={"": "cuda:7"})
    dispatch_model(pipe.vae, device_map={"": "cuda:7"})

    pipe.vae.enable_tiling()
    pipe.vae.enable_slicing()

elif per_gpu_mem >= 45:
    print(f"🚀 {per_gpu_mem:.0f} GB VRAM — loading full precision model")

    pipe = HunyuanVideoPipeline.from_pretrained(
        "hunyuanvideo-community/HunyuanVideo",
        torch_dtype=torch.bfloat16,
        token=HF_TOKEN,
    )
    pipe.to("cuda")
    pipe.vae.enable_tiling()

else:
    print(f"💾 {per_gpu_mem:.0f} GB VRAM — loading quantized (int4) model (~14 GB)")

    from diffusers.quantizers import PipelineQuantizationConfig

    pipeline_quant_config = PipelineQuantizationConfig(
        quant_backend="bitsandbytes_4bit",
        quant_kwargs={
            "load_in_4bit": True,
            "bnb_4bit_quant_type": "nf4",
            "bnb_4bit_compute_dtype": torch.bfloat16,
        },
        components_to_quantize=["transformer"],
    )

    pipe = HunyuanVideoPipeline.from_pretrained(
        "hunyuanvideo-community/HunyuanVideo",
        quantization_config=pipeline_quant_config,
        torch_dtype=torch.bfloat16,
        token=HF_TOKEN,
    )
    pipe.enable_model_cpu_offload()
    pipe.vae.enable_tiling()

print("✅ HunyuanVideo loaded!")

🚀 85 GB VRAM — loading full precision model


Loading pipeline components...: 100%|██████████| 7/7 [00:06<00:00,  1.10it/s]


✅ HunyuanVideo loaded!


In [8]:
import time

# ✏️ EDIT YOUR PROMPT HERE
prompt = """
A Pomeranian sprints through a park between trees and benches, weaving playfully. Dynamic tracking shot with smooth steadycam, quick but clean parallax, light motion blur, sharp focus on the dog. Sun rays through leaves, cinematic contrast, realistic fur simulation and paw impacts on grass. 4K, 60fps for smooth motion, 6 seconds. No text, no logos.
"""

# ✏️ SETTINGS
NUM_FRAMES = 122        # 4k+1 rule: 33 frames / 15fps ≈ 2 sec
HEIGHT = 480           # Conservative first — bump once balance is confirmed
WIDTH = 848
STEPS = 30             # 30 is the recommended default
FPS = 15

# Clear any leftover memory before generation
torch.cuda.empty_cache()

print(f"🎬 Generating {NUM_FRAMES} frames at {WIDTH}x{HEIGHT}...")
start = time.time()

output = pipe(
    prompt=prompt,
    num_frames=NUM_FRAMES,
    height=HEIGHT,
    width=WIDTH,
    num_inference_steps=STEPS,
).frames[0]

elapsed = time.time() - start
print(f"✅ Done in {elapsed:.0f}s ({elapsed/60:.1f} min)")

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['logos .']


🎬 Generating 122 frames at 848x480...


100%|██████████| 30/30 [10:32<00:00, 21.09s/it]


✅ Done in 657s (10.9 min)


In [9]:
import base64
from IPython.display import HTML, display

output_path = "hunyuan_output.mp4"
export_to_video(output, output_path, fps=FPS)
print(f"✅ Saved to {output_path}")

with open(output_path, "rb") as f:
    video_b64 = base64.b64encode(f.read()).decode()

display(HTML(f"""
<video width="{WIDTH}" height="{HEIGHT}" controls autoplay loop>
  <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
</video>
"""))

✅ Saved to hunyuan_output.mp4


In [10]:
for i in range(torch.cuda.device_count()):
    peak = torch.cuda.max_memory_allocated(i) / 1e9
    print(f"GPU {i}: peak {peak:.1f} GB")

torch.cuda.empty_cache()
print("\n🧹 CUDA cache cleared")

GPU 0: peak 47.0 GB

🧹 CUDA cache cleared
